> 在线讲义：[第 21 章 · LLaMA 配方：Pre-norm + RMSNorm + SwiGLU + RoPE 怎么拼到一起](https://wikieden.github.io/beyond-transformer/lessons/21-llama.html) · 本书《超越 Transformer》

# 第 21 章 · LLaMA 配方：Pre-norm + RMSNorm + SwiGLU + RoPE 怎么拼到一起

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/21-llama.html) · 仓库里 `beyond-transformer/notes/lessons/21-llama.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 换零件 | 一层的参数 | `layer_params(**kw)`、`STEPS` | 111,840 → 110,784 |
| 第二步 学习率 | 余弦到 10% | `cosine_to_10pct(t)` | 第 100,000 步 3.0e-5 |

一层的参数怎么变；训练超参（AdamW β₂ = 0.95、余弦到 10%、权重衰减 0.1、裁剪 1.0、warmup 2000）。
只算一个玩具宽度的一层；给真实模型（GPT-2、Llama、Qwen）算参数量并对官方数是里程碑 T-M1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
D = 96

**函数 `layer_params`**：一层（注意力 + FFN + 两个 norm）的参数个数。ffn：relu（d → 4d → d）或 swiglu（d → 8d/3 取 64 倍数，三个矩阵）。

In [ ]:
def layer_params(d=D, ffn="relu", norm="layer", bias=True, pos="learned"):
    """一层（注意力 + FFN + 两个 norm）的参数个数。ffn：relu（d → 4d → d）或 swiglu（d → 8d/3 取 64 倍数，三个矩阵）。"""
    attn = 4 * d * d + (4 * d if bias else 0)
    if ffn == "relu":
        f = 4 * d
        mlp = 2 * d * f + ((f + d) if bias else 0)
    else:
        f = int(math.ceil(8 * d / 3 / 64) * 64)
        mlp = 3 * d * f
    norms = 2 * (2 * d if norm == "layer" else d)
    return {"注意力": attn, "FFN": mlp, "norm": norms}

In [ ]:
STEPS = [
    ("原版（Post-LN、LayerNorm、ReLU FFN、有 bias）", dict(ffn="relu", norm="layer", bias=True)),
    ("换成 Pre-norm + RMSNorm", dict(ffn="relu", norm="rms", bias=True)),
    ("FFN 换成 SwiGLU（d_ff = 8d/3）", dict(ffn="swiglu", norm="rms", bias=True)),
    ("去掉所有 bias", dict(ffn="swiglu", norm="rms", bias=False)),
]
TRAIN = {"优化器": "AdamW，β₁ = 0.9、β₂ = 0.95", "学习率调度": "余弦，最后降到峰值的 10%", "权重衰减": "0.1", "梯度裁剪": "1.0", "warmup": "2000 步"}

**函数 `cosine_to_10pct`**：LLaMA §2.3：warmup 后余弦降到峰值的 10%。

In [ ]:
def cosine_to_10pct(t, peak=3e-4, warm=2000, total=100_000):
    """LLaMA §2.3：warmup 后余弦降到峰值的 10%。"""
    if t < warm:
        return peak * t / warm
    p = (t - warm) / (total - warm)
    return 0.1 * peak + 0.5 * (1 + math.cos(math.pi * p)) * 0.9 * peak

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print(f"第一步 一个零件一个零件地换（d = {D} 的一层）")
for name, kw in STEPS:
    p = layer_params(**kw)
    print(f"  {name}：{p}，合计 {sum(p.values()):,}")
print("  位置编码：原版加在嵌入上的正弦 / 学习位置表 → 每层注意力里的 RoPE（不占参数）")
print("\n第二步 训练超参")
for k, v in TRAIN.items():
    print(f"  {k}：{v}")
for t in (0, 1000, 2000, 51000, 100000):
    print(f"  t = {t:6d}：lr = {cosine_to_10pct(t):.2e}")

## 3. 代码里才看得到的

- **只数了一层的矩阵**：嵌入层、输出层、最后的 norm 都没算进来；完整地给真实配置数参数是里程碑 T-M1。
- **d = 64 时对不齐**：8/3·64 = 170.7，取 64 的倍数是 192，SwiGLU 反而多 12% 参数；所以这里选了 d = 96。实际模型（d = 4096）误差在 1% 以内。

## 4. 自己改一改

- 把 `D` 改回 64，四步的合计怎么变？SwiGLU 那一步多了多少？
- 把 `cosine_to_10pct` 的最低点改成 0（降到 0），最后 1000 步的学习率差多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 老车换四样：norm 挪前面换 RMS，FFN 换 SwiGLU，位置换 RoPE，bias 全去掉；参数几乎不变。
> 训练照前几章：AdamW 贝塔二零点九五，warmup 两千，余弦降到一成，衰减零点一，裁剪一。
> 小模型多训数据：训练贵一点，推理一直便宜。